# Lección 3.3 — Optimización: el excursionista en la niebla (descenso de gradiente)

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.1 (perillas) y Lección 3.2 (función de pérdida y su paisaje).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar el **descenso de gradiente** con la analogía del excursionista en la niebla.
- Entender qué es la **pendiente** (el *gradiente*) y por qué hay que moverse **en sentido contrario**.
- Ejecutar paso a paso el algoritmo y **ver** el camino que sigue en el paisaje de la pérdida.
- Reconocer por qué un valle en forma de **cañón** hace lento el entrenamiento, y cómo lo arregla el escalado.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. Trae los datos de la heladería, el modelo (la recta) y la pérdida MSE de las lecciones anteriores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)

rng = np.random.default_rng(0)
temperatura = rng.uniform(14, 36, 30).round(1)
ventas = (8 * temperatura - 60 + rng.normal(0, 15, 30)).round()

def modelo(x, w, b):
    return w * x + b

def mse(real, pred):
    return np.mean((pred - real) ** 2)

print("✅ Todo listo")

---
## 1. 📖 La intuición: el excursionista en la niebla

Estás en lo alto de una montaña y cae una **niebla espesa**. No ves nada a más de un metro. Quieres llegar al **fondo del valle**, donde está el refugio. ¿Qué haces?

```
   niebla ░░░░░░░░░░░░░░░░░░░░░░░░░░
          ░░░░  🧍 ← tú (no ves el valle)
          ░░░░░░  ╲
                   ╲   1. Siente con los pies hacia dónde BAJA el suelo.
                    ╲  2. Da un paso en esa dirección.
                     ╲__ 3. Repite hasta que el suelo quede plano.
                        ╲____🏠 refugio (el mínimo)
```

No necesitas un mapa: te basta con **sentir la inclinación del suelo justo donde estás** y dar pasos cuesta abajo.

En nuestro problema:
- La **montaña** es el **paisaje de la pérdida** (Lección 3.2).
- **Tu posición** son los valores actuales de las **perillas** (`w`, `b`).
- **Bajar** = cambiar las perillas para que la pérdida disminuya.
- El **refugio** son las mejores perillas.

Este algoritmo se llama **descenso de gradiente** (*gradient descent*) y entrena casi todos los modelos modernos: regresiones, redes neuronales, ChatGPT…

### ✋ Ejercicio 1 — ¿Qué necesita saber el excursionista?
En cada paso, ¿qué información **necesita** el excursionista para decidir hacia dónde moverse?
- a) Un mapa completo de la montaña
- b) La inclinación del suelo bajo sus pies
- c) La altura de la cima más alta
- d) Dónde está el refugio

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** Solo la **inclinación local**. Es la gran ventaja del método: nunca necesita ver todo el paisaje (que con millones de perillas sería imposible de dibujar o explorar, ¿recuerdas la explosión de combinaciones de la Lección 3.1?). Solo pregunta *"¿hacia dónde baja desde aquí?"*.
</details>

---
## 2. 📖 El concepto: la pendiente te dice hacia dónde bajar

Empecemos con **una sola perilla**: dejamos `b = -56` fijo y solo movemos `w`. El paisaje es la curva en forma de tazón de la Lección 3.2.

¿Cómo "siente con los pies" la computadora? Mueve la perilla **un pasito minúsculo** y mira si la pérdida sube o baja:

$$\text{pendiente} \approx \frac{\text{pérdida}(w + 0.001) - \text{pérdida}(w)}{0.001}$$

- **Pendiente positiva** (+): si subo `w`, la pérdida **sube** → me conviene **bajar** `w`.
- **Pendiente negativa** (−): si subo `w`, la pérdida **baja** → me conviene **subir** `w`.
- **Pendiente ≈ 0**: el suelo está plano → **llegué al fondo**.

A esa pendiente los matemáticos la llaman **derivada** o, cuando hay varias perillas, **gradiente**. No necesitas calcularla a mano: la computadora lo hace por ti.

In [ ]:
def perdida_w(w):
    # Pérdida cuando solo movemos w (b fijo en -56)
    return mse(ventas, modelo(temperatura, w, -56))

def pendiente(f, w, pasito=0.001):
    # "Sentir el suelo": mover un pasito y ver cuánto cambia la pérdida
    return (f(w + pasito) - f(w)) / pasito

for w in [3, 8, 12]:
    print(f"w = {w:>2}  →  pérdida = {perdida_w(w):>8.0f}   pendiente = {pendiente(perdida_w, w):>+9.0f}")

### 🔮 Ejercicio 2 — Lee la pendiente
Con los números de arriba:
1. En `w = 3`, ¿debes subir o bajar `w`?
2. En `w = 12`, ¿subir o bajar?
3. En `w = 8` la pendiente es mucho más pequeña que en los otros dos puntos. ¿Qué te dice eso?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Pendiente **negativa** → la pérdida baja si subes `w` → **subir** `w`.
2. Pendiente **positiva** → **bajar** `w`.
3. Que estás **en el fondo** o muy cerca: el suelo es casi plano (pendiente ≈ −2, contra miles en los otros puntos).

Regla de oro: **muévete siempre en sentido contrario a la pendiente.**
</details>

---
## 3. 📖 Cómo funciona: la regla del paso

Juntando todo, cada paso del excursionista es:

$$w_{\text{nuevo}} = w_{\text{actual}} - \text{tasa de aprendizaje} \times \text{pendiente}$$

| Pieza | Qué es | En la analogía |
|---|---|---|
| **pendiente** | Hacia dónde sube la pérdida (y qué tan inclinado está) | La inclinación bajo tus pies |
| el signo **menos** | Ir en sentido **contrario**, cuesta abajo | Caminar hacia donde baja el suelo |
| **tasa de aprendizaje** (*learning rate*) | Un número pequeño que decide el **tamaño del paso** | El largo de tus zancadas |

Fíjate en algo bonito: como el paso es **tasa × pendiente**, cuando el terreno es empinado das pasos largos y, al acercarte al fondo (pendiente pequeña), tus pasos se acortan **solos**.

Vamos a caminar. Empezamos en `w = 1` (muy lejos del fondo).

In [ ]:
TASA = 0.0005        # tamaño de las zancadas (✏️ lo cambiarás en el ejercicio 4)
PASOS = 10
w = 1.0              # punto de partida

camino = [w]
print(" paso |    w    | pendiente | pérdida")
for paso in range(1, PASOS + 1):
    p = pendiente(perdida_w, w)
    w = w - TASA * p                 # ⭐ LA REGLA DEL DESCENSO DE GRADIENTE
    camino.append(w)
    print(f" {paso:>4} | {w:>7.3f} | {p:>+9.0f} | {perdida_w(w):>8.1f}")

ws = np.linspace(0, 14, 200)
plt.plot(ws, [perdida_w(v) for v in ws], color="gray", lw=2, label="Paisaje de la pérdida")
plt.plot(camino, [perdida_w(v) for v in camino], "o-", color="tab:red", label="Camino del excursionista")
plt.plot(camino[0], perdida_w(camino[0]), "ks", ms=10, label="Inicio")
plt.xlabel("Perilla w"); plt.ylabel("Pérdida (MSE)"); plt.legend(); plt.grid(alpha=.3); plt.show()

### 🔮 Ejercicio 3 — Mira las zancadas
1. Observa la tabla: ¿cuánto se mueve `w` en el primer paso? ¿Y en el quinto?
2. ¿Por qué los pasos se hacen más cortos si la `TASA` nunca cambió?
3. ¿En cuántos pasos llega prácticamente al fondo (`w` ya casi no cambia)?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. En el primer paso `w` salta de 1 a **≈ 5.98** (casi 5 unidades). En el quinto, apenas se mueve **0.03**.
2. Porque el paso es **tasa × pendiente**, y la pendiente se va haciendo pequeña al acercarse al fondo. El algoritmo **frena solo**.
3. En unos **5 pasos** ya está en `w ≈ 7.99` y la pérdida deja de bajar (≈ 199).
</details>

### ✏️ Ejercicio 4 — Primer contacto con la tasa de aprendizaje
Vuelve a la celda anterior, cambia `TASA` y ejecútala otra vez. Prueba en este orden y anota qué pasa:

| TASA | ¿Qué observas? |
|---|---|
| `0.0001` | |
| `0.0005` | (ya lo viste) |
| `0.0013` | |
| `0.0016` | |

✍️ *Mis observaciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **`0.0001`** → pasos **muy cortos**: tras 10 pasos sigue lejos del fondo (`w` ≈ 6.5, pérdida ≈ 1,800). Llega, pero tarda mucho.
- **`0.0005`** → llega en pocos pasos. 👍
- **`0.0013`** → **zigzag**: salta de un lado al otro del valle (1 → 13.96 → 2.93 → 12.31…), pero cada salto es un poco más pequeño y termina llegando, muy despacio.
- **`0.0016`** → **diverge**: cada salto lo deja *más arriba* que el anterior (w llega a −73) y la pérdida se dispara a millones. El excursionista sale volando del valle.

La tasa de aprendizaje es el hiperparámetro más importante del entrenamiento. Le dedicamos toda la **Lección 3.4**.
</details>

---
## 4. 📖 Con dos perillas: una brújula de dos agujas

Con `w` y `b` a la vez, el excursionista necesita saber la pendiente **en cada dirección**: cuánto sube la pérdida si muevo `w` y cuánto si muevo `b`. Esas dos pendientes juntas forman el **gradiente**: una flecha que apunta **cuesta arriba**. Nosotros caminamos en la dirección **opuesta**.

En vez del "pasito" numérico, para la MSE existe una fórmula exacta y rápida. No hace falta memorizarla; fíjate solo en su lógica:

In [ ]:
def gradiente(w, b, x, y):
    error = modelo(x, w, b) - y            # + si predigo de más, − si predigo de menos
    pendiente_w = 2 * np.mean(error * x)   # ¿cuánto cambia la pérdida si muevo w?
    pendiente_b = 2 * np.mean(error)       # ¿cuánto cambia la pérdida si muevo b?
    return pendiente_w, pendiente_b

# Comprobación: la fórmula exacta coincide con el método del "pasito"
w0, b0 = 3.0, 20.0
f_w = lambda w: mse(ventas, modelo(temperatura, w, b0))
f_b = lambda b: mse(ventas, modelo(temperatura, w0, b))
print("Fórmula exacta :", np.round(gradiente(w0, b0, temperatura, ventas), 1))
print("Método 'pasito':", np.round([pendiente(f_w, w0), pendiente(f_b, b0)], 1))

Lee `pendiente_b`: si en promedio el modelo **predice de más** (error positivo), la pendiente es positiva y el paso **baja** `b`. Si predice de menos, lo sube. ¡Es lo que hiciste a mano en la Lección 3.1!

> 📝 En redes neuronales, esta fórmula se calcula automáticamente con la **retropropagación** (*backpropagation*, Lección 9.4). La idea es la misma: un gradiente por perilla.

### 📖 Un pequeño truco antes de caminar
Para que el valle sea más fácil de recorrer (en vez del cañón diagonal de la Lección 3.2), mediremos la temperatura como **grados por encima de 25 °C** (un día de 30 °C es `+5`; uno de 20 °C es `−5`). Así `b` pasa a significar *"helados vendidos en un día de 25 °C"*. En la Sección 5 verás por qué ayuda tanto.

In [ ]:
x = temperatura - 25          # grados por encima de 25 °C

def descenso_gradiente(x, y, w, b, tasa, pasos):
    historia = [(w, b, mse(y, modelo(x, w, b)))]
    for _ in range(pasos):
        gw, gb = gradiente(w, b, x, y)
        w, b = w - tasa * gw, b - tasa * gb          # ⭐ un paso cuesta abajo en cada perilla
        historia.append((w, b, mse(y, modelo(x, w, b))))
    return np.array(historia)

def dibujar_camino(x, y, historia, rango_w, rango_b, titulo):
    W, B = np.meshgrid(np.linspace(*rango_w, 120), np.linspace(*rango_b, 120))
    Z = ((W[..., None] * x + B[..., None] - y) ** 2).mean(axis=-1)
    plt.contourf(W, B, np.log10(Z), levels=30, cmap="viridis")
    plt.plot(historia[:, 0], historia[:, 1], "o-", color="tab:red", ms=3, lw=1)
    plt.plot(*historia[0, :2], "ws", ms=9, mec="k", label="Inicio")
    plt.plot(*historia[-1, :2], "r*", ms=15, label="Final")
    plt.xlabel("Perilla w"); plt.ylabel("Perilla b"); plt.title(titulo); plt.legend(); plt.show()

historia = descenso_gradiente(x, ventas, w=0, b=0, tasa=0.015, pasos=300)
print(f"Inicio : w = {historia[0, 0]:.2f}, b = {historia[0, 1]:.1f}, pérdida = {historia[0, 2]:.0f}")
print(f"Final  : w = {historia[-1, 0]:.2f}, b = {historia[-1, 1]:.1f}, pérdida = {historia[-1, 2]:.0f}")
dibujar_camino(x, ventas, historia, (-2, 17), (-10, 200), "Camino del descenso de gradiente (300 pasos)")

### ✋ Ejercicio 5 — Lee el camino
1. ¿En qué valores de `w` y `b` terminó? Traduce `w` y `b` a lenguaje de la heladería.
2. ¿Coincide `w` con el que encontraste a mano en la Lección 3.1?
3. Observa la forma del camino: ¿va en línea recta al fondo o hace algo raro al principio?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Termina en **`w ≈ 7.99`** y **`b ≈ 144`**: *"por cada grado extra se venden unos 8 helados más, y en un día de 25 °C se venden unos 144"*. La pérdida bajó de ≈ 25,800 a ≈ **199**.
2. Sí: el `w ≈ 8` es el mismo de la Lección 3.1. (Y si deshaces el truco, `b` original = 144 − 8 × 25 ≈ **−56**, el mismo intercepto que vimos en la 3.2.)
3. Al principio **zigzaguea** en la dirección de `w` (el valle es más empinado en esa dirección) mientras avanza poco a poco en la dirección de `b`. El gradiente apunta hacia donde el terreno es más empinado, no necesariamente directo al refugio.
</details>

### 🔮 Ejercicio 6 — La curva de la pérdida
Ahora dibujamos la **pérdida en cada paso**. Antes de ejecutar, dibuja en papel cómo crees que se verá: ¿una línea recta hacia abajo? ¿una curva? ¿con escalones?

✍️ *Mi dibujo / descripción:*

In [ ]:
plt.plot(historia[:, 2], lw=2)
plt.yscale("log")
plt.xlabel("Paso"); plt.ylabel("Pérdida (MSE, escala log)"); plt.title("La pérdida en cada paso")
plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Baja **muy rápido al principio** y luego se va **aplanando** hasta quedarse quieta en ≈ 199: la forma de "codo" típica de un entrenamiento sano. Esta gráfica se llama **curva de aprendizaje** y es tu principal herramienta para diagnosticar entrenamientos (Lección 3.7).
</details>

---
## 5. 📖 ¿Qué pasa si el valle es un cañón?

En la Lección 3.2 viste que, con la temperatura original (14 a 36 °C), el valle es un **cañón largo y estrecho**. ¿Por qué importa?

El gradiente apunta hacia donde el terreno es **más empinado**. En un cañón, eso es casi siempre **hacia las paredes**, no a lo largo del cañón hacia el refugio. El excursionista rebota de pared a pared y avanza muy lento.

Además, como las paredes son muy empinadas, hay que usar zancadas **muy cortas** para no salir volando, lo que lo hace todavía más lento.

### 🔮 Ejercicio 7
Vamos a entrenar con la temperatura **original** (sin restar 25), con la mayor tasa que no explota (0.0007), durante **1,000 pasos** (más de 3 veces más que antes). ¿Crees que llegará al mismo fondo (pérdida ≈ 199)?

✍️ *Mi apuesta:*

In [ ]:
hist_canon = descenso_gradiente(temperatura, ventas, w=0, b=0, tasa=0.0007, pasos=1000)
print(f"Temperatura original, 1000 pasos → w = {hist_canon[-1, 0]:.2f}, b = {hist_canon[-1, 1]:.1f}, pérdida = {hist_canon[-1, 2]:.0f}")
dibujar_camino(temperatura, ventas, hist_canon, (0, 14), (-200, 100), "Temperatura original: el cañón (1000 pasos)")

<details><summary><b>👀 Ver respuesta</b></summary>

**No llega.** Tras 1,000 pasos la pérdida sigue en ≈ **375** (contra 199 del fondo), `w` va en ≈ 6.15 y `b` apenas en **≈ −5** cuando debería estar en −56. El excursionista llegó rápido al fondo del cañón, pero luego avanza **a paso de tortuga** a lo largo de él. Necesitaría decenas de miles de pasos.

Restar 25 a la temperatura **enderezó** el valle: dejó de ser un cañón diagonal y quedó alineado con los ejes, así que cada perilla puede bajar "por su cuenta" y el gradiente apunta mucho más hacia el fondo. En la práctica se hace algo parecido con **todas** las características: **escalarlas** (restar la media y dividir por la desviación estándar, Lección 2.3).

👉 **Por eso los modelos que se entrenan con descenso de gradiente (regresiones, redes neuronales) casi siempre necesitan datos escalados.** Los árboles (Lección 5.4) no lo necesitan porque no usan gradientes.
</details>

---
## 6. 📖 Las perillas del propio entrenamiento

El descenso de gradiente tiene sus propios **hiperparámetros**. Los dos que ya conociste:

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Tasa de aprendizaje** (`learning_rate`, `lr`, `eta0`) | Tamaño de cada paso | Más rápido… hasta que zigzaguea y **diverge** | Más estable pero **muy lento** | 0.01 (con datos escalados); 0.001 en redes con Adam |
| **Número de pasos** (`max_iter`, iteraciones) | Cuánto camina el excursionista | Llega más cerca del fondo (y más tiempo de cómputo) | Se detiene antes de llegar | Hasta que la pérdida deje de bajar |
| **Punto de inicio** (inicialización) | Desde dónde empieza a caminar | — | — | Ceros o valores pequeños al azar (importa mucho en la Lección 3.6) |

En la Lección 3.4 los exploraremos a fondo, junto con las **épocas** y el **tamaño de lote**.

### ✏️ Ejercicio 8 — Llega al fondo con el mínimo esfuerzo
Con la temperatura centrada (`x`), busca la `TASA` que llegue a pérdida **≤ 200 en la menor cantidad de pasos** posible. Ojo: si la tasa es demasiado grande, explota.

In [ ]:
TASA, PASOS = 0.015, 300      # ✏️ prueba, por ejemplo, 0.005, 0.01, 0.02, 0.021...

h = descenso_gradiente(x, ventas, w=0, b=0, tasa=TASA, pasos=PASOS)
print(f"TASA={TASA}, PASOS={PASOS} → pérdida final = {h[-1, 2]:,.1f}")
llego = np.where(h[:, 2] <= 200)[0]
print(f"Llegó a ≤ 200 en el paso {llego[0]}" if len(llego) else "❌ No llegó a ≤ 200")

<details><summary><b>👀 Ver respuesta</b></summary>

Con `0.005` no llega en 300 pasos; con `0.01` llega en el paso **248**; con `0.015`, en el **164**; con **`0.02`**, en el **136**. Con `0.0203` ya zigzaguea tanto que no llega, y con **`0.021`** el entrenamiento **diverge** (la pérdida se dispara a valores astronómicos). Fíjate en que la mejor tasa está **justo antes** del punto en que explota: es una idea que usaremos en la Lección 3.4.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida crece hasta `inf` o `nan` | Tasa de aprendizaje demasiado grande | Dividir la tasa entre 10 |
| La pérdida baja, pero con una lentitud desesperante | Tasa demasiado pequeña **o** datos sin escalar (cañón) | Escalar las características; luego subir la tasa |
| Moverse **a favor** de la pendiente (signo `+` en vez de `−`) | Error de programación clásico: subes la montaña | Revisa la regla: `w = w − tasa × pendiente` |
| Confundir el gradiente con la pendiente de la recta (`w`) | Son cosas distintas: `w` es una **perilla**; el gradiente es la **inclinación de la pérdida** | Recuerda: el gradiente vive en el paisaje, no en la gráfica de los datos |
| Esperar llegar **exactamente** al mínimo | El algoritmo se acerca cada vez más, pero los pasos se encogen | Detente cuando la pérdida deje de mejorar de forma apreciable |

### 🔮 Ejercicio 9 — El error del signo
¿Qué pasará si alguien escribe `w = w + TASA * p` en lugar de `w = w − TASA * p`? Predice y luego compruébalo.

✍️ *Tu respuesta:*

In [ ]:
w = 7.0
for paso in range(5):
    w = w + 0.0005 * pendiente(perdida_w, w)         # ❌ signo equivocado
    print(f"paso {paso + 1}: w = {w:>10.2f}   pérdida = {perdida_w(w):>14,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El excursionista **sube** la montaña: aunque empieza muy cerca del fondo (`w = 7`), `w` se aleja cada vez más rápido y la pérdida crece sin control. Un signo cambia "aprender" por "desaprender".
</details>

---
## 8. 📋 Ficha del algoritmo

| | Descenso de gradiente |
|---|---|
| **Qué es** | Un método de **optimización**: encuentra las perillas que minimizan la pérdida |
| **Intuición** | Un excursionista en la niebla que siente la pendiente y da pasos cuesta abajo |
| **Regla** | `perilla = perilla − tasa_de_aprendizaje × pendiente` (para cada perilla) |
| **Hiperparámetros clave** | Tasa de aprendizaje, número de pasos/épocas, tamaño de lote (Lección 3.5), inicialización |
| **✅ Fortalezas** | Funciona igual con 2 que con miles de millones de perillas · Solo necesita información local · Es la base de todo el deep learning |
| **❌ Debilidades** | Sensible a la tasa de aprendizaje · Lento en valles tipo cañón (hay que escalar) · Puede quedarse en mínimos locales (Lección 3.6) |
| **Lo usan** | Regresión lineal y logística (`SGDRegressor`, `SGDClassifier`), redes neuronales (Módulo 9), boosting (Lección 6.3, en otra forma) |
| **No lo usan** | Árboles de decisión (Lección 5.4), KNN (Lección 5.3) |

## 9. 🧠 Resumen en 5 puntos
1. El **descenso de gradiente** baja por el paisaje de la pérdida sintiendo solo la **pendiente local**, como un excursionista en la niebla.
2. La **pendiente** (gradiente) dice hacia dónde **sube** la pérdida; por eso nos movemos en sentido **contrario**.
3. Cada paso: `perilla = perilla − tasa × pendiente`. Cerca del fondo los pasos se acortan solos.
4. La **tasa de aprendizaje** decide el tamaño de los pasos: muy chica = lento; muy grande = zigzag o explosión.
5. En valles tipo **cañón** el algoritmo es lentísimo: **escalar los datos** convierte el cañón en un tazón.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** La pendiente de la pérdida respecto a `w` es **+50**. ¿Qué hace el descenso de gradiente con `w`?
- a) La sube · b) La baja · c) La deja igual · d) La multiplica por 50

**2.** Con tasa = 0.1 y pendiente = −20, ¿cuánto cambia la perilla en ese paso?
- a) −2 · b) +2 · c) −20 · d) +0.1

**3.** ¿Por qué los pasos se acortan cerca del mínimo, aunque la tasa no cambie?
- a) Porque el algoritmo se cansa · b) Porque la pendiente se hace pequeña · c) Porque la pérdida se vuelve negativa · d) Porque cambian los datos

**4.** La pérdida se vuelve `nan` después de unos pasos. ¿Qué es lo primero que pruebas?
- a) Subir la tasa · b) Bajar la tasa · c) Más pasos · d) Cambiar el signo de la regla

**5.** ¿Cuál de estos modelos **no** se entrena con descenso de gradiente?
- a) Regresión logística · b) Red neuronal · c) Árbol de decisión · d) `SGDRegressor`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Pendiente positiva = la pérdida sube si subes `w`, así que la baja.
2. **b)** −0.1 × (−20) = **+2**.
3. **b)** El paso es tasa × pendiente.
4. **b)** `nan` casi siempre es una tasa demasiado grande que hizo explotar los números.
5. **c)** El árbol elige divisiones de forma voraz (Lección 5.4).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** El excursionista es "miope": solo ve la pendiente bajo sus pies. ¿Qué podría salir mal si la montaña tuviera **varios valles** de distintas profundidades?

**B.** Un modelo tiene dos características: *edad* (de 18 a 90) y *salario mensual* (de 5,000 a 200,000). Tu compañero lo entrena con descenso de gradiente sin escalar y dice que "tarda una eternidad y luego explota si sube la tasa". Explícale qué pasa usando la analogía del cañón.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Podría bajar al **primer valle que encuentre** y quedarse ahí, aunque haya otro más profundo al lado: no tiene forma de saberlo. Es el problema de los **mínimos locales**, que veremos en la Lección 3.6 (junto con la buena noticia de que, en la práctica, a menudo importa menos de lo que parece).

**B.** Como el salario tiene números miles de veces más grandes, la pérdida es **muchísimo más empinada** en la dirección de su perilla que en la de la edad: el paisaje es un cañón extremo. Para no salir volando en la dirección empinada necesita una tasa diminuta, y con esa tasa la perilla de la edad casi no se mueve. **Solución: escalar ambas características** (Lección 2.3).
</details>

### 10.3 🧩 Práctica integradora: entrena tú el modelo del taxi 🚕

En la Lección 3.1 ajustaste a mano el modelo `tarifa = w · km + b`. Ahora lo vas a entrenar con descenso de gradiente, paso a paso.

**Paso 1 ▶️ — Datos y truco del centrado.** Mediremos la distancia como "km por encima de 10" (`x_taxi`), así `b` significa *"tarifa de un viaje de 10 km"*.

In [ ]:
rng_taxi = np.random.default_rng(5)
km = rng_taxi.uniform(1, 20, 50).round(1)
tarifa = (35 + 9 * km + rng_taxi.normal(0, 6, 50)).round()
x_taxi = km - 10

print(f"Pérdida con w=0, b=0: {mse(tarifa, modelo(x_taxi, 0, 0)):,.0f}")

**Paso 2 🔮 — Siente el suelo.** Empezamos en `w = 0, b = 0` (el modelo dice "todos los viajes cuestan $0"). Antes de ejecutar: ¿el gradiente de `b` será positivo o negativo? ¿Hacia dónde se moverá `b`?

✍️ *Mi predicción:*

In [ ]:
gw, gb = gradiente(0, 0, x_taxi, tarifa)
print(f"pendiente_w = {gw:+.1f}   ·   pendiente_b = {gb:+.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

`pendiente_b` es **negativa** (≈ −253): el modelo predice $0 y los viajes cuestan ≈ $126 en promedio, o sea **predice de menos**. El descenso de gradiente **subirá** `b`. También `pendiente_w` es negativa: hay que subir `w`.
</details>

**Paso 3 ✋ — Un paso a mano.** Con `tasa = 0.01`, calcula en papel los nuevos `w` y `b` tras **un** paso. Luego compruébalo.

In [ ]:
tasa = 0.01
w1, b1 = 0 - tasa * gw, 0 - tasa * gb
print(f"Tras 1 paso: w = {w1:.2f}, b = {b1:.2f}, pérdida = {mse(tarifa, modelo(x_taxi, w1, b1)):,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

`w` = 0 − 0.01 × (−658.3) ≈ **6.58** y `b` = 0 − 0.01 × (−252.8) ≈ **2.53**. La pérdida baja de ≈ 18,700 a ≈ 15,200 con un solo paso.
</details>

**Paso 4 ✏️ — Deja caminar al excursionista.** Usa `descenso_gradiente` con `w=0, b=0`. Elige `TASA` y `PASOS` hasta que la pérdida baje de **30**. Si la pérdida explota, baja la tasa.

In [ ]:
TASA, PASOS = 0.01, 50        # ✏️

h_taxi = descenso_gradiente(x_taxi, tarifa, w=0, b=0, tasa=TASA, pasos=PASOS)
print(f"Final: w = {h_taxi[-1, 0]:.2f}, b = {h_taxi[-1, 1]:.2f}, pérdida = {h_taxi[-1, 2]:.1f}")
plt.plot(h_taxi[:, 2]); plt.yscale("log"); plt.xlabel("Paso"); plt.ylabel("Pérdida"); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con `TASA = 0.01` y 50 pasos se queda lejísimos (pérdida ≈ 2,077): `w` ya está cerca (≈ 9.4), pero `b` va en ≈ 79. Opciones que sí llegan al fondo (≈ **28.9**): `TASA = 0.01` con 300 pasos, o `TASA = 0.02` con 200. Con `0.03` zigzaguea y necesita ~500 pasos, y desde **`0.035`** el entrenamiento diverge. Al final `w ≈ 9.04` y `b ≈ 124.2`.
</details>

**Paso 5 ✋ — Compara y traduce.**
1. ¿Coincide tu `w` con el de `np.polyfit` de la Lección 3.1 (≈ 9.04)?
2. Tu `b` es la tarifa de un viaje de **10 km**. ¿Cómo obtienes el banderazo (la tarifa a 0 km)?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Sí: el descenso de gradiente llega **al mismo lugar** que la fórmula directa. Para una recta existe esa fórmula, pero para una red neuronal no: por eso el descenso de gradiente es tan importante.
2. A 10 km se pagan ≈ $124.2, y cada km cuesta ≈ $9.04; quitando 10 km: 124.2 − 10 × 9.04 ≈ **$33.8**, el mismo banderazo de la Lección 3.1.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** 3Blue1Brown, *"Gradient descent, how neural networks learn"* (inglés, con subtítulos).
- 📺 **Video:** StatQuest, *"Gradient Descent, Step-by-Step"*.
- 🎮 **Interactivo:** *distill.pub/2017/momentum* ("Why Momentum Really Works"): juega con la tasa de aprendizaje en un valle tipo cañón.

---
⬅️ **Anterior:** [3.2 Función de pérdida: medir qué tan equivocado está el modelo](3.2-funcion-de-perdida.ipynb) · ➡️ **Siguiente:** [3.4 Hiperparámetros del entrenamiento: tasa de aprendizaje, épocas, tamaño de lote e iteraciones](3.4-hiperparametros-del-entrenamiento.ipynb)